# Build a static installed-package capability catalog

This notebook builds a real, queryable catalog from explicitly selected installed Python distributions. It reads distribution metadata with `importlib.metadata`, reads bounded `.py` files, and parses them with `ast`; it does **not** import or execute target package modules. Observed package releases and operations remain separate from derived representations, labels, and blocking keys.

The default two-package build is intentionally small enough for a Kaggle CPU session. Increase a shard's package list and resource budgets only after inspecting the error and size summaries. Outputs are written beneath `/kaggle/working/capability_catalog/<SHARD_ID>` as normalized SQLite, Parquet mirrors, JSONL errors, a hierarchy table, and a content-hashed manifest.

For `REPO_MODE = "clone"`, enable Internet in the Kaggle notebook settings. For `REPO_MODE = "upload"`, attach the repository as a Kaggle Dataset and set `UPLOADED_REPO_ROOT`.

In [ ]:
import os
from pathlib import Path

# ---------------------------- editable configuration ----------------------------
IS_KAGGLE = Path("/kaggle/working").is_dir()
WORKING_ROOT = Path("/kaggle/working") if IS_KAGGLE else Path.cwd() / ".notebook_working"

# Choose "clone" for a GitHub checkout or "upload" for an attached Kaggle Dataset.
REPO_MODE = "clone"
REPO_URL = "https://github.com/Amarel-Taylor-Scott/this-already-exists-dont-rebuild-it.git"
REPO_REF = "main"
CLONE_DEPTH = 1
LOCAL_REPO_ROOT = WORKING_ROOT / "this-already-exists-dont-rebuild-it"
UPLOADED_REPO_ROOT = Path("/kaggle/input/this-already-exists-dont-rebuild-it")

# Install specifications and distribution metadata names are intentionally separate.
# Each bounded package batch gets an immutable shard identity and output directory.
SHARD_ID = os.environ.get("CAPABILITY_CATALOG_SHARD_ID", "shard-00000").strip()
TARGET_PACKAGE_SPECS = ("packaging>=24,<27", "python-dateutil>=2.9,<3")
TARGET_DISTRIBUTIONS = ("packaging", "python-dateutil")
MAX_DISTRIBUTIONS = 2

# Static source budgets. Files are sorted before these limits are applied.
MAX_FILES_PER_DISTRIBUTION = 100
MAX_BYTES_PER_FILE = 1_000_000
MAX_TOTAL_BYTES_PER_DISTRIBUTION = 12_000_000
INCLUDE_PRIVATE_SYMBOLS = True

# Hierarchy depth: package=1; subpackages/modules/symbols add one level each.
MAX_HIERARCHY_DEPTH = 8
EXAMPLE_LIMIT = 12
FTS_RESULT_LIMIT = 10
OUTPUT_ROOT = WORKING_ROOT / "capability_catalog" / SHARD_ID

assert REPO_MODE in {"clone", "upload"}
assert SHARD_ID and "/" not in SHARD_ID and "\\" not in SHARD_ID and SHARD_ID not in {".", ".."}
assert TARGET_DISTRIBUTIONS and MAX_DISTRIBUTIONS > 0
assert MAX_FILES_PER_DISTRIBUTION >= 0
assert MAX_BYTES_PER_FILE >= 0 and MAX_TOTAL_BYTES_PER_DISTRIBUTION >= 0
assert MAX_HIERARCHY_DEPTH >= 1
print({"is_kaggle": IS_KAGGLE, "repo_mode": REPO_MODE, "shard_id": SHARD_ID, "output_root": str(OUTPUT_ROOT)})

In [ ]:
import shlex
import subprocess
import sys

# Runtime dependencies for the catalog and its Parquet exports.
!pip install -q "packaging>=24" "pandas>=2" "pyarrow>=14"
target_specs = " ".join(shlex.quote(spec) for spec in TARGET_PACKAGE_SPECS)
!pip install -q {target_specs}

if REPO_MODE == "clone":
    if not (LOCAL_REPO_ROOT / "src/existing_code_reuse/ingest.py").is_file():
        LOCAL_REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
        local_repo_text = str(LOCAL_REPO_ROOT)
        !git clone --filter=blob:none --depth {CLONE_DEPTH} --branch {REPO_REF} {REPO_URL} {local_repo_text}
    repo_root = LOCAL_REPO_ROOT
else:
    repo_root = UPLOADED_REPO_ROOT
    if not (repo_root / "src/existing_code_reuse/ingest.py").is_file():
        input_root = Path("/kaggle/input") if IS_KAGGLE else Path.cwd()
        matches = sorted(input_root.glob("**/src/existing_code_reuse/ingest.py"))
        if len(matches) == 1:
            repo_root = matches[0].parents[2]
        else:
            raise FileNotFoundError(
                f"Set UPLOADED_REPO_ROOT to the attached repository; found {len(matches)} candidates"
            )

source_root = repo_root / "src"
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

from existing_code_reuse.ingest import IngestConfig, ingest_installed_distributions
from existing_code_reuse.models import (
    DependencyRecord,
    DerivedSignal,
    OperationRecord,
    PackageReleaseRecord,
    RepresentationRecord,
    canonical_json,
    digest_value,
)
from existing_code_reuse.storage import read_catalog, search_fts, write_catalog

print({"repo_root": str(repo_root), "source_root": str(source_root)})

## Build the bounded static catalog

`IngestConfig` applies deterministic file and byte budgets independently to each distribution. The extractor records package metadata, requirements, top-level public/private functions and classes, and direct class methods. Source-read, decode, parse, metadata, path, and budget failures remain in `catalog.errors`.

In [ ]:
from datetime import datetime, timezone
import time

selected_distributions = tuple(TARGET_DISTRIBUTIONS[:MAX_DISTRIBUTIONS])
ingest_config = IngestConfig(
    max_files_per_distribution=MAX_FILES_PER_DISTRIBUTION,
    max_bytes_per_file=MAX_BYTES_PER_FILE,
    max_total_bytes_per_distribution=MAX_TOTAL_BYTES_PER_DISTRIBUTION,
    include_private=INCLUDE_PRIVATE_SYMBOLS,
)
build_started_at = datetime.now(timezone.utc)
start = time.perf_counter()
catalog = ingest_installed_distributions(selected_distributions, config=ingest_config)
build_seconds = time.perf_counter() - start

if not catalog.packages:
    raise RuntimeError(f"No installed distributions were cataloged; errors={catalog.errors}")
assert all(operation.evidence_level == "static_source" for operation in catalog.operations)
print({
    "selected_distributions": selected_distributions,
    "build_seconds": round(build_seconds, 3),
    **catalog.counts(),
})

## Persist normalized and analysis-friendly artifacts

SQLite is the canonical local artifact. Parquet tables mirror each typed record family for Kaggle analysis. `hierarchy.parquet` makes the package → subpackage → module → class/function → method structure explicit without changing the underlying observed operation records.

In [ ]:
from dataclasses import fields
import hashlib
import json
import pandas as pd

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
sqlite_path = write_catalog(OUTPUT_ROOT / "catalog.sqlite", catalog)

def json_safe(value):
    if isinstance(value, tuple):
        return [json_safe(item) for item in value]
    if isinstance(value, list):
        return [json_safe(item) for item in value]
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    return value

def records_frame(records, record_type):
    columns = [field.name for field in fields(record_type)]
    rows = [json_safe(record.to_dict()) for record in records]
    return pd.DataFrame(rows, columns=columns)

table_specs = {
    "packages": (catalog.packages, PackageReleaseRecord),
    "operations": (catalog.operations, OperationRecord),
    "representations": (catalog.representations, RepresentationRecord),
    "signals": (catalog.signals, DerivedSignal),
    "dependencies": (catalog.dependencies, DependencyRecord),
}
frames = {}
for table_name, (records, record_type) in table_specs.items():
    frame = records_frame(records, record_type)
    frame.to_parquet(OUTPUT_ROOT / f"{table_name}.parquet", index=False)
    frames[table_name] = frame

errors_path = OUTPUT_ROOT / "errors.jsonl"
with errors_path.open("w", encoding="utf-8") as error_file:
    for error in catalog.errors:
        error_file.write(canonical_json(error) + "\n")

In [ ]:
# Build an explicit, content-addressed hierarchy projection from observed operations.
hierarchy_columns = [
    "node_id", "node_type", "parent_id", "depth", "package_id",
    "package_name", "module", "qualified_name", "operation_id",
    "kind", "visibility",
]
nodes = {}
package_nodes = {}
module_nodes = {}
class_nodes = {}

def hierarchy_id(node_type, package_id, qualified_name):
    value = digest_value({
        "schema": "catalog-hierarchy-node-v1",
        "node_type": node_type,
        "package_id": package_id,
        "qualified_name": qualified_name,
    })
    return "node:" + value.removeprefix("sha256:")

def add_node(*, node_type, parent_id, depth, package_id, package_name, module="",
             qualified_name="", operation_id="", kind="", visibility=""):
    if depth > MAX_HIERARCHY_DEPTH or (parent_id and parent_id not in nodes):
        return None
    node_id = hierarchy_id(node_type, package_id, qualified_name or module or package_name)
    nodes.setdefault(node_id, {
        "node_id": node_id, "node_type": node_type, "parent_id": parent_id,
        "depth": depth, "package_id": package_id, "package_name": package_name,
        "module": module, "qualified_name": qualified_name,
        "operation_id": operation_id, "kind": kind, "visibility": visibility,
    })
    return node_id

package_by_id = {package.package_id: package for package in catalog.packages}
for package in catalog.packages:
    package_nodes[package.package_id] = add_node(
        node_type="package", parent_id=None, depth=1, package_id=package.package_id,
        package_name=package.name, qualified_name=package.normalized_name,
    )

for package_id, module in sorted({(item.package_id, item.module) for item in catalog.operations}):
    package = package_by_id[package_id]
    parts = tuple(part for part in module.split(".") if part)
    parent_id = package_nodes[package_id]
    depth = 1
    for prefix_length in range(2, len(parts)):
        prefix = ".".join(parts[:prefix_length])
        depth += 1
        parent_id = add_node(
            node_type="subpackage", parent_id=parent_id, depth=depth,
            package_id=package_id, package_name=package.name, module=prefix,
            qualified_name=prefix,
        )
        if parent_id is None:
            break
    if parent_id is not None:
        module_nodes[(package_id, module)] = add_node(
            node_type="module", parent_id=parent_id, depth=depth + 1,
            package_id=package_id, package_name=package.name, module=module,
            qualified_name=module,
        )

for operation in catalog.operations:
    if operation.kind != "class":
        continue
    parent_id = module_nodes.get((operation.package_id, operation.module))
    if parent_id is None:
        continue
    depth = nodes[parent_id]["depth"] + 1
    node_id = add_node(
        node_type="class", parent_id=parent_id, depth=depth,
        package_id=operation.package_id, package_name=operation.package_name,
        module=operation.module, qualified_name=operation.qualified_name,
        operation_id=operation.operation_id, kind=operation.kind,
        visibility=operation.visibility,
    )
    class_nodes[(operation.package_id, operation.module, operation.qualified_name)] = node_id

for operation in catalog.operations:
    if operation.kind in {"function", "async_function"}:
        parent_id = module_nodes.get((operation.package_id, operation.module))
        node_type = "function"
    elif operation.kind in {"method", "async_method"}:
        class_name = operation.qualified_name.rsplit(".", 1)[0]
        parent_id = class_nodes.get((operation.package_id, operation.module, class_name))
        node_type = "method"
    else:
        continue
    if parent_id is None:
        continue
    add_node(
        node_type=node_type, parent_id=parent_id, depth=nodes[parent_id]["depth"] + 1,
        package_id=operation.package_id, package_name=operation.package_name,
        module=operation.module, qualified_name=operation.qualified_name,
        operation_id=operation.operation_id, kind=operation.kind,
        visibility=operation.visibility,
    )

hierarchy_df = pd.DataFrame(
    sorted(nodes.values(), key=lambda row: (row["package_id"], row["depth"], row["node_type"], row["qualified_name"])),
    columns=hierarchy_columns,
)
hierarchy_path = OUTPUT_ROOT / "hierarchy.parquet"
hierarchy_df.to_parquet(hierarchy_path, index=False)
hierarchy_df.groupby(["depth", "node_type"], dropna=False).size().rename("count").reset_index()

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as source_file:
        for chunk in iter(lambda: source_file.read(1 << 20), b""):
            digest.update(chunk)
    return "sha256:" + digest.hexdigest()

artifact_paths = sorted(path for path in OUTPUT_ROOT.iterdir() if path.name != "manifest.json")
manifest = {
    "schema": "static-installed-capability-catalog-v1",
    "shard_id": SHARD_ID,
    "output_root": str(OUTPUT_ROOT),
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "build_started_at": build_started_at.isoformat(),
    "build_seconds": build_seconds,
    "repository": {"mode": REPO_MODE, "root": str(repo_root), "ref": REPO_REF},
    "selected_distributions": list(selected_distributions),
    "limits": {
        "max_files_per_distribution": MAX_FILES_PER_DISTRIBUTION,
        "max_bytes_per_file": MAX_BYTES_PER_FILE,
        "max_total_bytes_per_distribution": MAX_TOTAL_BYTES_PER_DISTRIBUTION,
        "include_private_symbols": INCLUDE_PRIVATE_SYMBOLS,
        "max_hierarchy_depth": MAX_HIERARCHY_DEPTH,
    },
    "counts": catalog.counts(),
    "hierarchy_nodes": int(len(hierarchy_df)),
    "artifacts": {
        path.name: {"bytes": path.stat().st_size, "digest": sha256_file(path)}
        for path in artifact_paths
    },
    "safety": {
        "target_modules_imported_by_ingester": False,
        "target_symbols_executed_by_ingester": False,
        "extraction_method": "python_ast_v1",
    },
}
manifest_path = OUTPUT_ROOT / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print({"manifest": str(manifest_path), "artifact_count": len(manifest["artifacts"])})

## Inspect observed records, derivations, and preserved failures

The following cells calculate summaries from the current build. Empty output in the committed notebook is intentional; no counts are hard-coded.

In [ ]:
from IPython.display import display

count_summary = pd.DataFrame(
    [{"record_type": key, "count": value} for key, value in catalog.counts().items()]
)
package_summary = frames["packages"][
    ["name", "version", "installed_file_count", "evidence_level", "source_digest"]
]
operation_summary = (
    frames["operations"]
    .groupby(["package_name", "kind", "visibility"], dropna=False)
    .size()
    .rename("count")
    .reset_index()
)
display(count_summary)
display(package_summary)
display(operation_summary)
display(hierarchy_df.groupby("node_type").size().rename("count").reset_index())

if catalog.errors:
    errors_df = pd.DataFrame(catalog.errors)
    display(errors_df.groupby(["stage", "error_type"], dropna=False).size().rename("count").reset_index())
    display(errors_df.head(EXAMPLE_LIMIT))
else:
    print("No errors were recorded for this bounded build.")

In [ ]:
operation_examples = frames["operations"][
    ["package_name", "module", "qualified_name", "kind", "signature",
     "visibility", "relative_path", "source_digest"]
].head(EXAMPLE_LIMIT)
representation_examples = frames["representations"][
    ["operation_id", "plane", "text", "generator", "generator_version"]
].head(EXAMPLE_LIMIT)
signal_examples = frames["signals"][
    ["operation_id", "signal_kind", "namespace", "value", "confidence",
     "evidence_fields", "generator_version"]
].head(EXAMPLE_LIMIT)
display(operation_examples)
display(representation_examples)
display(signal_examples)

## Query a subpackage and a symbol

This demonstration queries the persisted normalized operation table for the `packaging.licenses` subpackage and the symbol name `parse`, then searches independent representation planes through SQLite FTS5. Change these values to inspect another installed package.

In [ ]:
import sqlite3

QUERY_SUBPACKAGE = "packaging.licenses"
QUERY_SYMBOL = "parse"

with sqlite3.connect(sqlite_path) as connection:
    subpackage_hits = pd.read_sql_query(
        """
        SELECT package_name, module, qualified_name, kind, signature, visibility, operation_id
        FROM operations
        WHERE module = ? OR module LIKE ?
        ORDER BY module, qualified_name, operation_id
        LIMIT ?
        """,
        connection,
        params=(QUERY_SUBPACKAGE, QUERY_SUBPACKAGE + ".%", EXAMPLE_LIMIT),
    )
    symbol_hits = pd.read_sql_query(
        """
        SELECT package_name, module, qualified_name, kind, signature, operation_id
        FROM operations
        WHERE qualified_name = ? OR qualified_name LIKE ?
        ORDER BY package_name, module, qualified_name, operation_id
        LIMIT ?
        """,
        connection,
        params=(QUERY_SYMBOL, "%." + QUERY_SYMBOL, EXAMPLE_LIMIT),
    )

fts_hits = search_fts(sqlite_path, "parse AND version", limit=FTS_RESULT_LIMIT)
fts_df = pd.DataFrame([
    {
        "representation_id": hit.representation_id,
        "operation_id": hit.operation_id,
        "plane": hit.plane,
        "rank": hit.rank,
        "text": hit.text,
    }
    for hit in fts_hits
])
display(subpackage_hits)
display(symbol_hits)
display(fts_df)

In [ ]:
# Read the canonical artifact back and verify row counts and content hashes before handoff.
round_trip = read_catalog(sqlite_path)
assert round_trip.counts() == catalog.counts()
for artifact_name, artifact in manifest["artifacts"].items():
    artifact_path = OUTPUT_ROOT / artifact_name
    assert artifact_path.stat().st_size == artifact["bytes"]
    assert sha256_file(artifact_path) == artifact["digest"]

print("Catalog build verified. Artifacts:")
for path in sorted(OUTPUT_ROOT.iterdir()):
    print(f"- {path.name}: {path.stat().st_size:,} bytes")

## Scaling the next run

Do not scale one notebook process to 10,000 packages. Orchestrate 10,000+ packages as many bounded, immutable shards: assign each frozen package/version batch a unique `SHARD_ID`, keep per-distribution file and byte limits, run shards independently, and merge their manifests or indexes only in a separate downstream step. Never overwrite a published shard. Within one shard, add pinned install specifications and matching distribution names, then raise `MAX_DISTRIBUTIONS` only while `errors.jsonl`, build time, SQLite size, and operation counts remain acceptable. `MAX_HIERARCHY_DEPTH` affects only the hierarchy projection; it never changes the extracted operation evidence. Preserve every shard's `manifest.json` with its generated artifacts.